In [ ]:
from tqdm import tqdm
import aiomysql

with open(".secret-db-pass", "r") as file:
    conn = await aiomysql.connect(host='pi.tgr.rs', port=3306, user='memesearch', password=file.read(), db='memesearch')

memes, page_size, limit = [], 500, 1000000

async with conn.cursor() as cur:
    await cur.execute("SELECT MAX(id) FROM memes")
    result = await cur.fetchone()
    print(f"got {result[0]}, limit {limit}")

    for page_id in tqdm(range(0, min(limit, result[0]) // page_size)):
        start = 1 + page_id * page_size
        await cur.execute("SELECT hash, embedding_bin FROM memes WHERE id >= %s AND id < %s AND embedding_bin IS NOT NULL", (start, start + page_size))
        memes += await cur.fetchall()

conn.close()
print(f"read {len(memes)} memes")

with open("/Users/enovikov11/Desktop/data-mac/memesearch/memes.bin", "wb") as file:
    for meme in memes:
        file.write(bytes.fromhex(meme[0][0:64]))
        file.write(meme[1])

In [ ]:
from IPython.display import display
from openai import OpenAI
from PIL import Image
from tqdm import tqdm
import numpy as np
import aiomysql
import struct
import base64
import faiss

with open(".secret-oai", "r") as file:
    client = OpenAI(api_key=file.read())

with open(".secret-db-pass", "r") as file:
    conn = await aiomysql.connect(host='localhost', port=3306, user='memesearch', password=file.read(), db='memesearch')

def describe(path):
    with open(path, 'rb') as file:
        encoded_image = "data:image/jpeg;base64," + base64.b64encode(file.read()).decode('utf-8')

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user",
            "content": [
                { "type": "image_url", "image_url": { "url": encoded_image, "detail": "high" } },
                { "type": "text", "text": "Опиши мем" }
        ]}],
        temperature=1, max_tokens=2048, top_p=1, frequency_penalty=0, presence_penalty=0, response_format={"type": "text"}
    )

    return response.choices[0].message.content

def embed(description):
    return client.embeddings.create(input=[description], model="text-embedding-3-large").data[0].embedding

async def build_index(page_size = 500):
    memes = []

    async with conn.cursor() as cur:
        await cur.execute("SELECT MAX(id) FROM memes")
        result = await cur.fetchone()

        for page_id in tqdm(range(0, result[0] // page_size)):
            start = 1 + page_id * page_size
            await cur.execute("SELECT hash, embedding_bin FROM memes WHERE id >= %s AND id < %s AND embedding_bin IS NOT NULL", (start, start + page_size))
            memes += await cur.fetchall()

    hashes, embeddings_bin = zip(*memes)
    embeddings = np.array([struct.unpack('3072d', data) for data in embeddings_bin], dtype=np.float32)

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)

    return hashes, index

hashes, index = None, None

def search(query, results=10):
    query_vector = np.array([embed(query)], dtype=np.float32)
    distances, indexes = index.search(query_vector, results)

    images = [hashes[i] for i in indexes[0]]

    for image in images:
        display(Image.open(f"/Users/enovikov11/Code/data-local/meme-search/memes/{image}"))

In [ ]:
# find . -type f -print0 | xargs -0 sha256sum > sha256.sum

# await cur.execute("SELECT id, description FROM memes WHERE id >= %s AND id < %s AND embedding_bin IS NOT NULL", (start, start + page_size))

# page = await cur.fetchall()
# bins = [(struct.pack('3072d', *embedding), id) for id, embedding in page]

# await cur.executemany("UPDATE memes SET embedding_bin = %s WHERE id = %s", bins)
# await conn.commit()